In [24]:
from diffusers import UNet2DModel
import torch
from diffusers import DDPMScheduler

try: 
    import torch_xla.core.xla_model as xm
except: pass

from models import load_model
from utils.utils import *
from utils.utils_data import *
from utils.utils_ebm import *
from utils.utils_baselines import *
import matplotlib.pyplot as plt
import numpy as np

import os
os.environ['XRT_TPU_CONFIG'] = 'localservice;0;localhost:51011'

device = xm.xla_device()

In [2]:
# Load Diff Model
model_id = "google/ddpm-cifar10-32"
diff = UNet2DModel.from_pretrained(model_id).to(device)

# Load EBM
ebm = load_ebm(f'/home/sunaybhat/data/models/ebms/ebm_cifar10_45k.pt', 128).to(device)

### Diff Scheduler and EBM transforms

In [30]:
scheduler = DDPMScheduler.from_config(model_id)
scheduler.config['num_train_timesteps'] = 125  # Reduced number of diffusion steps
scheduler.config['beta_start'] = .00001  # Starting noise level
scheduler.config['beta_end'] = 0.00002  # You may need to adjust this based on performance
scheduler.config['beta_schedule'] = 'linear'  # Consider experimenting with 'cosine' or custom schedules
# Additional configurations here if needed

scheduler.save_config("my_scheduler")
scheduler = DDPMScheduler.from_config("my_scheduler")

fwd_transform = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
inverse_transform = transforms.Normalize((-1, -1, -1), (2, 2, 2))

/home/sunaybhat/.local/lib/python3.8/site-packages/diffusers/configuration_utils.py:244: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a scheduler, please use <class 'diffusers.schedulers.scheduling_ddpm.DDPMScheduler'>.from_pretrained(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)


### Data

In [31]:
train_data = torchvision.datasets.CIFAR10(root='/home/sunaybhat/data', train=True, download=True, transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4)
batch,_ = next(iter(train_loader))

noise_npy = torch.from_numpy(np.load('/home/sunaybhat/data/Poisons/Narcissus/size=32_eps=8/best_noise_lab5.npy'))
batch_poisoned = batch + noise_npy


Files already downloaded and verified


## Compare Diff Params

In [33]:
img_idx = np.random.choice(batch.shape[0], 1)[0]
sample_image = batch[img_idx]
sample_image_poisoned = batch_poisoned[img_idx]

# Initialize an empty list to store the images
samples = [sample_image_poisoned.cpu().squeeze().permute(1, 2, 0)]


for num_steps in [25,50,100,125,200]:

  scheduler.config['num_train_timesteps'] = num_steps  # Reduced number of diffusion steps
  scheduler.save_config("my_scheduler")
  scheduler = DDPMScheduler.from_config("my_scheduler")

  sample = batch

  for i, t in enumerate(tqdm(scheduler.timesteps)):
    # print(t)
    # 1. predict noise residual
    with torch.no_grad(): residual = diff(sample, t).sample

      xm.mark_step()

    # 2. compute less noisy image and set x_t -> x_t-1
    sample = scheduler.step(residual, t, sample).prev_sample

  # Store the sample in the list
  image_clipped = torch.clamp(sample, -1, 1).cpu().squeeze().permute(1, 2, 0)
  samples.append(image_clipped)

# Create a figure with a number of subplots equal to the number of samples
fig, axs = plt.subplots(1, len(samples), figsize=(50, 100))

# Plot each sample in its own subplot
for i, sample in enumerate(samples):
    axs[i].imshow(sample)
    axs[i].axis('off')


  0%|          | 0/25 [00:00<?, ?it/s]


RuntimeError: /pytorch/xla/torch_xla/csrc/aten_xla_bridge.cpp:74 : Check failed: xtensor 
*** Begin stack trace ***
	tsl::CurrentStackTrace()
	torch_xla::bridge::GetXlaTensor(at::Tensor const&)
	torch_xla::XLANativeFunctions::addmm(at::Tensor const&, at::Tensor const&, at::Tensor const&, c10::Scalar const&, c10::Scalar const&)
	
	at::_ops::addmm::redispatch(c10::DispatchKeySet, at::Tensor const&, at::Tensor const&, at::Tensor const&, c10::Scalar const&, c10::Scalar const&)
	
	
	at::_ops::addmm::call(at::Tensor const&, at::Tensor const&, at::Tensor const&, c10::Scalar const&, c10::Scalar const&)
	at::native::linear(at::Tensor const&, at::Tensor const&, c10::optional<at::Tensor> const&)
	
	at::_ops::linear::call(at::Tensor const&, at::Tensor const&, c10::optional<at::Tensor> const&)
	
	PyCFunction_Call
	_PyObject_MakeTpCall
	_PyEval_EvalFrameDefault
	
	_PyEval_EvalFrameDefault
	_PyEval_EvalCodeWithName
	_PyFunction_Vectorcall
	
	PyObject_Call
	_PyEval_EvalFrameDefault
	_PyEval_EvalCodeWi

In [29]:
scheduler.config

FrozenDict([('num_train_timesteps', 200),
            ('beta_start', 0.0001),
            ('beta_end', 0.02),
            ('beta_schedule', 'linear'),
            ('trained_betas', None),
            ('variance_type', 'fixed_large'),
            ('clip_sample', True),
            ('prediction_type', 'epsilon'),
            ('thresholding', False),
            ('dynamic_thresholding_ratio', 0.995),
            ('clip_sample_range', 1.0),
            ('sample_max_value', 1.0),
            ('timestep_spacing', 'leading'),
            ('steps_offset', 0),
            ('rescale_betas_zero_snr', False),
            ('_class_name', 'DDPMScheduler'),
            ('_diffusers_version', '0.26.2')])

In [14]:
sample_image.shape

torch.Size([3, 32, 32])

In [7]:
sample_image.shape

torch.Size([3, 32, 32])

In [5]:
import matplotlib.pyplot as plt

# Initialize an empty list to store the images
samples = [image_1.cpu().squeeze().permute(1, 2, 0)]

sample = image_1.to(gen.device)

for i, t in enumerate(tqdm(scheduler.timesteps)):
  # print(t)
  # 1. predict noise residual
  with torch.no_grad():
    residual = gen(sample, t).sample

  print("Beta at timestep {}: {}".format(i, scheduler.get_beta(t)))


  # 2. compute less noisy image and set x_t -> x_t-1
  sample = scheduler.step(residual, t, sample).prev_sample

  # Store the sample in the list
  image_clipped = torch.clamp(sample, -1, 1).cpu().squeeze().permute(1, 2, 0)
  if i % 10 == 0:
    samples.append(image_clipped)

# Create a figure with a number of subplots equal to the number of samples
fig, axs = plt.subplots(1, len(samples), figsize=(50, 100))

# Plot 10 even spaces sampled

for i, sample in enumerate(samples):
  axs[i].imshow(sample)  # replace with the appropriate function to display your images
  axs[i].axis('off')  # remove axes for clarity



  0%|          | 0/100 [00:00<?, ?it/s]


AttributeError: 'DDPMScheduler' object has no attribute 'get_beta'